<a href="https://colab.research.google.com/github/emmakelly7/ds2002-fa26/blob/main/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df["revenue"] = df["qty"] * df["price"]
total_revenue = df["revenue"].sum()
total_units = df["qty"].sum()

print("Total revenue:", total_revenue)
print("Total units:", total_units)

Total revenue: 8520.0
Total units: 783


In [3]:
df.shape

(400, 5)

This dataset has 400 orders and 5 variables, which are vendor_id, category, price, qty, and the revenue column just created. It also has a total revenue of $8,520 and 783 total units sold.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [4]:
category_revenue = (
    df.groupby("category", as_index=False)["revenue"].sum()
)
category_revenue["share_percent"] = (
    100 * category_revenue["revenue"] / df["revenue"].sum()
)

category_revenue = category_revenue.sort_values(
    "revenue", ascending=False
)

category_revenue

,category,revenue,share_percent
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


I used category_revenue to organize the revenues, and then used share_percent to calculate the percentage of the revenue that each category constituted. I also used sort and ascending=False to output the information highest to lowest. I found that Food created the most revenue at 4,293 dollars and that was about 50.38% of total revenue. Then, the revenue Merch created was 1,771.50 dollars, which was about 20.79% of total revenue. Then, Drink had a revenue of 1,554 dollars, which was 18.24% of total revenue, and lastly, RainGear's revenue was 901.50 dollars which was 10.58%.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [5]:
vendor_names = df.groupby("vendor_id", as_index=False).agg(
    avg_revenue=("revenue", "mean"),
    order_count=("revenue", "count")
)

vendor_summary = vendor_names.sort_values("avg_revenue", ascending=False)

vendor_summary

,vendor_id,avg_revenue,order_count
0,V-01,22.595745,94
3,V-18,21.750000,108
1,V-05,20.580645,93
2,V-10,20.314286,105


To find the vendor with the highest average order revenue, I grouped by vendor_id and calculated the average revenue and number of orders for each vendor. Then, I sorted the vendors from highest to lowest average revenue. V-01 had the highest average order revenue at approximately 22.60 dollars with 94 orders. V-18 had the second highest average at 21.75 dollars across 108 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [6]:
merch = df[df["category"] == "Merch"].copy()
share = 100 * merch["revenue"].sum() / df["revenue"].sum()

round(share, 1)

np.float64(20.8)

I filtered it to make it only include orders in the Merch category. I then divided the total Merch revenue by the total revenue for all categories and multiplied by 100 to find the percentage. I found that Merch was about 20.8% of the total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [7]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on="vendor_id", how="left", validate="many_to_one")

print(joined.shape)
print(joined["revenue"].sum())

joined[joined["vendor_name"].isna()]

(400, 6)
8520.0


,vendor_id,category,qty,price,revenue,vendor_name
1,V-18,RainGear,1,12.0,12.0,NaN
2,V-18,Drink,3,4.5,13.5,NaN
4,V-18,Drink,3,7.5,22.5,NaN
5,V-18,Merch,1,24.0,24.0,NaN
6,V-18,Drink,1,24.0,24.0,NaN
...,...,...,...,...,...,...
384,V-18,Merch,1,4.5,4.5,NaN
386,V-18,Food,3,6.0,18.0,NaN
389,V-18,Food,1,4.5,4.5,NaN
395,V-18,Merch,1,12.0,12.0,NaN


**The unmatched vendor, and what I did about it:** _..._

The vendor_id that is not in the lookup is V-18. Using a left join kept the rows without a vendor_name, and then I used joined[joined["vendor_name"].isna()], and I found that there are 108 rows where V-18 is listed under vendor_id and NaN shows up under vendor_name. I kept the V-18 rows in the dataset because I thought the information was still significant, and I did not want to assign a random vendor_name to the vendor_id if I did not know what it was for.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [8]:
pivot = joined.pivot_table(
    index="vendor_id",
    columns="category",
    values="revenue",
    aggfunc="sum",
    margins=True
)

pivot

category,Drink,Food,Merch,RainGear,All
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


I created a pivot table and used index="vendor_id" to put the vendors down the side without getting rid of V-18, and then I used columns="category" to make the columns across the top. Then, I used values="revenue" to put the revenues in the cells. I used aggfunc="sum" to add all of the revenue sales, and then, I also used margins=True to add row and column totals so it reads as a report and not a grid of numbers.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [9]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(category_revenue['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a.) I would tell the vendors to concentrate more on food, as it created the largest revenue, with 4,293 dollars, which was about 50.38% of total revenue. I would also attempt to sway them away from focusing on RainGear, as it constituted only about 10.58% of total revenue at 901.50 dollars. Because of this, I think that the vendors might make a greater revenue if they dedicated more time and resources to food in place of RainGear. V-01 had the highest average order revenue at 22.60 dollars per order of the vendor_ids, so I would tell them to continue their current sales approach and tell other vendors to learn from them.

b.) I think the least trustworthy of my seven answers is Question 5 because I am less confident about the vendor_id in the orders that does not appear in the lookup. V-18 did not appear under vendor_names but was present under vendor_id. It had 108 orders, which is a significant amount of data that could not be connected to a vendor_name. I decided to keep the V-18 data because removing it would disrupt the total revenue information, but the missing vendor_name limits how confidently I can interpret the results by named vendor.